# Parrot on Chameleon

This notebook is a Chameleon/Trovi-oriented reproduction scaffold for **Parrot: Efficient Serving of LLM-based Applications with Semantic Variable**. It focuses on **resource reservation, server provisioning, Parrot installation, smoke testing, and benchmark execution setup**.

This notebook intentionally stops short of analysis and result plotting. The goal here is to make it straightforward to bring up the environment and run the workloads that will later feed the paper-style evaluation.

## What this notebook does

- reserves a suitable Chameleon node and a floating IP
- launches a server image on the reserved hardware
- connects to the server over SSH from the notebook
- installs Parrot and its dependencies
- records machine and software metadata for artifact reporting
- provides smoke tests for the Parrot Python package and service stack
- stages the benchmark repo layout and command hooks

## What you will still need to decide

- which Chameleon site and GPU node type to use
- whether you are targeting a single-GPU or multi-GPU reproduction
- which model weights you can access on the server
- whether exact paper-faithful workloads are feasible on the selected hardware

## Notes

Parrot's public installation guide expects Linux, CUDA 12.1, PyTorch 2.1, and a GPU with compute capability >= 7.0. The paper reports experiments on A100 and A6000 GPUs, but this notebook is written to degrade gracefully to the closest feasible Chameleon hardware.


## Reproduction Plan

The notebook follows this order:

1. configure Chameleon context
2. inspect available hardware and choose a target node type
3. create a lease with a floating IP
4. launch an Ubuntu server on the reserved node
5. attach the floating IP and verify connectivity
6. install Parrot, its dependencies, and optional benchmark dependencies
7. record environment metadata useful for artifact packaging
8. run smoke tests and benchmark entry points

If any step fails, stop there and capture the failure in the artifact notes rather than pushing forward blindly.


In [ ]:
from chi import context

# Opt into the newer notebook-friendly python-chi interface.
context.version = "1.0"

context.choose_site(default="CHI@TACC")
context.choose_project()


## Hardware Discovery

Parrot's paper used GPUs, so start by looking for GPU-capable nodes at your current site. The defaults below are intentionally conservative and should be adjusted after inspecting what Chameleon reports as available.


In [ ]:
from chi import hardware

# Display all nodes first if you want a broad overview.
# hardware.show_nodes()

# Common artifact workflow: filter for GPU nodes and inspect the table.
gpu_nodes = hardware.get_nodes(gpu=True)
print(f"Found {len(gpu_nodes)} GPU-capable nodes at the current site.")
hardware.show_nodes(gpu_nodes)


In [ ]:
# Set this after reviewing available hardware.
# Examples to consider are whatever Chameleon exposes at your site for recent NVIDIA GPUs.
node_type = None
minimum_hours = 6

if node_type is None:
    print("Pick a node_type from the table above before continuing.")
else:
    matches = hardware.get_nodes(node_type=node_type, filter_reserved=True)
    print(f"Available nodes for {node_type}: {len(matches)}")
    if matches:
        hardware.show_nodes(matches)
        start, end = matches[0].next_free_timeslot(minimum_hours=minimum_hours)
        print(f"Next free slot for an example node: {start} -> {end}")


## Reservation Parameters

The defaults below aim at a single-node run with one floating IP. Increase the duration if you expect long dependency builds or model downloads. For a multi-node or multi-GPU reproduction, adapt the reservation plan before submitting.


In [ ]:
from datetime import timedelta
import os

lease_name = f"{os.getenv('USER', 'user')}-parrot"
lease_hours = 8
image_name = "CC-Ubuntu24.04"
reserve_floating_ip = True

print({
    "lease_name": lease_name,
    "lease_hours": lease_hours,
    "image_name": image_name,
    "reserve_floating_ip": reserve_floating_ip,
})


In [ ]:
from chi import lease

if node_type is None:
    raise RuntimeError("Set node_type before creating the lease.")

my_lease = lease.Lease(
    name=lease_name,
    duration=timedelta(hours=lease_hours),
)
my_lease.add_node_reservation(amount=1, node_type=node_type)
if reserve_floating_ip:
    my_lease.add_fip_reservation(amount=1)

my_lease.submit(wait_for_active=True)
print(f"Lease status: {my_lease.status}")
print(my_lease)


## Launch a Server on the Reserved Hardware

This creates the bare-metal instance from the Chameleon image catalog. Launch times can vary substantially depending on node type and site load.


In [ ]:
from chi import server

reservation_id = my_lease.node_reservations[0]["id"]
my_server = server.Server(
    name=lease_name,
    reservation_id=reservation_id,
    image_name=image_name,
)
my_server.submit(wait_for_active=True)
print(f"Server status: {my_server.status}")
print(my_server)


## Associate a Floating IP

If the lease reserved a floating IP, attach it here. If `get_reserved_floating_ips()` returns an empty list, pause and allocate/associate an IP through the Chameleon dashboard or recreate the lease with `add_fip_reservation(amount=1)`.


In [ ]:
floating_ips = my_lease.get_reserved_floating_ips()
print("Reserved floating IPs:", floating_ips)

if not floating_ips:
    raise RuntimeError(
        "No reserved floating IPs found. Check the lease contents, wait for the lease to be ACTIVE, "
        "or allocate an IP manually from the Chameleon dashboard."
    )

floating_ip = floating_ips[0]
my_server.associate_floating_ip(floating_ip)
my_server.check_connectivity(host=floating_ip)
print(f"Connected via floating IP: {floating_ip}")


## Basic Machine Metadata

Capture the platform details early. These commands are useful later when writing the artifact description and when comparing deviations from the paper's original environment.


In [ ]:
machine_audit_cmds = [
    "hostname",
    "uname -a",
    "cat /etc/os-release",
    "nvidia-smi",
    "python3 --version",
    "gcc --version",
    "df -h",
    "free -h",
]

for cmd in machine_audit_cmds:
    print(f"\n===== {cmd} =====")
    print(my_server.execute(cmd))


## System Preparation

Parrot's public install instructions assume CUDA 12.1, PyTorch 2.1, and editable installs of both the top-level package and the bundled `3rdparty/vllm` tree. The commands below stage a virtual environment and clone the Parrot repo recursively.

Because dependency resolution for research systems can drift, these cells are deliberately explicit and log-heavy.


In [ ]:
bootstrap_script = r'''
set -euxo pipefail

export DEBIAN_FRONTEND=noninteractive
sudo apt-get update
sudo apt-get install -y \
    git git-lfs build-essential pkg-config cmake ninja-build \
    python3-dev python3-venv python3-pip \
    curl wget htop jq

cd /home/cc
if [ ! -d ParrotServe ]; then
  git clone --recursive https://github.com/microsoft/ParrotServe.git
fi

python3 -m venv /home/cc/parrot-venv
source /home/cc/parrot-venv/bin/activate
pip install --upgrade pip setuptools wheel

cd /home/cc/ParrotServe
# Mirror the published install guidance as closely as possible.
pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt
python -m pip install triton==2.1.0

cd /home/cc/ParrotServe/3rdparty/vllm
pip install -e .

cd /home/cc/ParrotServe
pip install -e .

# Optional benchmark extras from the public install note.
if [ -d /home/cc/ParrotServe/3rdparty/FastChat ]; then
  cd /home/cc/ParrotServe/3rdparty/FastChat
  pip install -e ".[model_worker,webui]" || true
fi
if [ -d /home/cc/ParrotServe/3rdparty/langchain/libs/langchain ]; then
  cd /home/cc/ParrotServe/3rdparty/langchain/libs/langchain
  pip install -e . || true
fi

cd /home/cc/ParrotServe
python - <<"PY"
import parrot
print("Imported parrot successfully:", getattr(parrot, "__file__", "unknown"))
PY
'''

print(my_server.execute(f"bash -lc {bootstrap_script!r}", timeout=7200))


## Environment Variables and Credentials

If the benchmark path you choose requires model downloads from Hugging Face or other registries, populate the relevant environment variables here before continuing. Avoid hard-coding secrets into the notebook if it will be shared through Trovi.


In [ ]:
# Fill these in only if needed for your chosen model workflow.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
MODEL_NAME = os.environ.get("PARROT_MODEL_NAME", "")

remote_env_lines = []
if HF_TOKEN:
    remote_env_lines.append(f"export HF_TOKEN={HF_TOKEN!r}")
if MODEL_NAME:
    remote_env_lines.append(f"export PARROT_MODEL_NAME={MODEL_NAME!r}")

if remote_env_lines:
    remote_env_script = "\n".join(remote_env_lines)
    print(my_server.execute(f"bash -lc {remote_env_script!r}"))
else:
    print("No extra remote environment variables were set from the notebook environment.")


## Record the Repository Layout

This is a lightweight sanity check before trying to start services or benchmarks. It also helps document which entry points are actually present in the checked-out revision.


In [ ]:
repo_inspect_script = r'''
set -euxo pipefail
source /home/cc/parrot-venv/bin/activate
cd /home/cc/ParrotServe
pwd
ls
find docs -maxdepth 2 -type f | sort | head -n 50
find examples -maxdepth 2 -type f | sort | head -n 50 || true
find benchmark -maxdepth 3 -type f | sort | head -n 100 || true
find sample_configs -maxdepth 3 -type f | sort | head -n 100 || true
'''

print(my_server.execute(f"bash -lc {repo_inspect_script!r}", timeout=1800))


## Python-Level Smoke Test

Start with the simplest possible check: can Python import Parrot and evaluate a tiny semantic function example on the frontend side? This validates the install before we involve the heavier serving stack.


In [ ]:
python_smoke_script = r'''
set -euxo pipefail
source /home/cc/parrot-venv/bin/activate
cd /home/cc/ParrotServe
python - <<"PY"
from parrot import P

@P.semantic_function()
def tell_me_a_joke(topic: P.Input, joke: P.Output):
    """Tell me a joke about {{topic}}: {{joke}}."""

print("Parrot frontend import and decorator smoke test passed.")
PY
'''

print(my_server.execute(f"bash -lc {python_smoke_script!r}", timeout=1800))


## Service Bring-Up Placeholder

The exact service-launch command may depend on which engine/backend config in the checked-out repo is the best match for your hardware. Use this cell to discover likely CLI entry points before committing to a long run.


In [ ]:
service_discovery_script = r'''
set -euxo pipefail
source /home/cc/parrot-venv/bin/activate
cd /home/cc/ParrotServe
python -m pip show parrot
python - <<"PY"
import pkgutil
import parrot
print("parrot package root:", parrot.__file__)
print("top-level submodules:")
for m in sorted({m.name.split('.')[0] for m in pkgutil.iter_modules(parrot.__path__)}):
    print(" -", m)
PY
find parrot -maxdepth 3 -type f | sort | head -n 200
'''

print(my_server.execute(f"bash -lc {service_discovery_script!r}", timeout=1800))


## Benchmark Staging

This section is intentionally notebook-friendly rather than paper-final. The idea is to isolate the commands you will eventually benchmark and to run them first in a smoke-test mode. Replace the placeholder command strings after inspecting the repo tree above.


In [ ]:
# Fill these with the actual entry points present in the checked-out ParrotServe revision.
# Examples might target an example app, a benchmark driver, or a manager/engine startup command.
manager_start_cmd = ""
engine_start_cmd = ""
benchmark_cmd = ""

for name, value in {
    "manager_start_cmd": manager_start_cmd,
    "engine_start_cmd": engine_start_cmd,
    "benchmark_cmd": benchmark_cmd,
}.items():
    print(f"{name}: {value or '[unset]'}")


In [ ]:
# Optional: create a working directory for logs and captured outputs.
prepare_run_dirs_script = r'''
set -euxo pipefail
mkdir -p /home/cc/parrot-runs/logs
mkdir -p /home/cc/parrot-runs/output
'''

print(my_server.execute(f"bash -lc {prepare_run_dirs_script!r}"))


In [ ]:
# Run the benchmark stack only after the command strings above are set.
if not manager_start_cmd or not engine_start_cmd or not benchmark_cmd:
    print("Set manager_start_cmd, engine_start_cmd, and benchmark_cmd before running this cell.")
else:
    launch_script = f"""
set -euxo pipefail
source /home/cc/parrot-venv/bin/activate
cd /home/cc/ParrotServe
nohup bash -lc {manager_start_cmd!r} > /home/cc/parrot-runs/logs/manager.log 2>&1 &
nohup bash -lc {engine_start_cmd!r} > /home/cc/parrot-runs/logs/engine.log 2>&1 &
sleep 20
bash -lc {benchmark_cmd!r} | tee /home/cc/parrot-runs/output/benchmark.txt
"""
    print(my_server.execute(f"bash -lc {launch_script!r}", timeout=7200))


## Pull Back Logs and Outputs

These cells download the most important logs and raw outputs into the notebook workspace for later packaging into a Trovi artifact.


In [ ]:
import os
from pathlib import Path

local_out = Path("parrot_artifact_out")
local_out.mkdir(exist_ok=True)
print(local_out.resolve())


In [ ]:
# Add or remove paths here based on what the run actually produced.
remote_paths = [
    "/home/cc/parrot-runs/logs/manager.log",
    "/home/cc/parrot-runs/logs/engine.log",
    "/home/cc/parrot-runs/output/benchmark.txt",
]

for remote_path in remote_paths:
    try:
        name = os.path.basename(remote_path)
        local_path = local_out / name
        my_server.download(remote_path, str(local_path))
        print(f"Downloaded {remote_path} -> {local_path}")
    except Exception as exc:
        print(f"Could not download {remote_path}: {exc}")


## Cleanup

When you are done, clean up the server and lease to avoid burning project resources. Leave these cells commented until you are certain you no longer need the instance.


In [ ]:
# Destructive operations: uncomment only when you are done.
# my_server.delete()
# my_lease.delete()
